# Netflix Movie Recommendation System Using CountVectorizer

## 🧠 Project Overview

This project aims to build a movie recommendation system — similar to Netflix — that suggests movies based on how similar they are in terms of their cast and crew.
In simple words, if you liked Avatar, this system tries to find other movies that have similar actors or directors.

We are using a dataset called tmdb_5000_credits.csv, which contains information about who acted in a movie and who directed it.

## 📂 Dataset Information

- File name: tmdb_5000_credits.csv
- Rows: ~4800 movies
- Columns:
- movie_id → Unique movie identifier
- title → Movie name
- cast → List of main actors
- crew → List of people who worked behind the scenes (like director, writer, etc.)

There are no missing values, and the data was clean and ready to use.

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re

In [18]:
data= pd.read_csv("dataset.csv")

In [19]:
data.head()

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [20]:
data.tail()

,movie_id,title,cast,crew
4798,9367,El Mariachi,"[{""cast_id"": 1, ""character"": ""El Mariachi"", ""c...","[{""credit_id"": ""52fe44eec3a36847f80b280b"", ""de..."
4799,72766,Newlyweds,"[{""cast_id"": 1, ""character"": ""Buzzy"", ""credit_...","[{""credit_id"": ""52fe487dc3a368484e0fb013"", ""de..."
4800,231617,"Signed, Sealed, Delivered","[{""cast_id"": 8, ""character"": ""Oliver O\u2019To...","[{""credit_id"": ""52fe4df3c3a36847f8275ecf"", ""de..."
4801,126186,Shanghai Calling,"[{""cast_id"": 3, ""character"": ""Sam"", ""credit_id...","[{""credit_id"": ""52fe4ad9c3a368484e16a36b"", ""de..."
4802,25975,My Date with Drew,"[{""cast_id"": 3, ""character"": ""Herself"", ""credi...","[{""credit_id"": ""58ce021b9251415a390165d9"", ""de..."


In [21]:
data.columns

Index(['movie_id', 'title', 'cast', 'crew'], dtype='object')

In [22]:
data.isnull().sum()

movie_id    0
title       0
cast        0
crew        0
dtype: int64

In [23]:
data.shape

(4803, 4)

In [24]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movie_id  4803 non-null   int64 
 1   title     4803 non-null   object
 2   cast      4803 non-null   object
 3   crew      4803 non-null   object
dtypes: int64(1), object(3)
memory usage: 150.2+ KB


## 🧩 Step-by-Step Implementation

### 1️⃣ Understanding the Data

Each row in the file represents one movie.
The cast and crew columns are written in a JSON-like format (like a list of small dictionaries).

So we first extracted the top 3 actor names and the director name for each movie.

In [25]:
def top_3_actors(cast_actor):
    cast=json.loads(cast_actor)
    return [actor["name"] for actor in cast[:3]]

data["top_3_actors"]= data["cast"].apply(top_3_actors)

In [26]:
def top_3_crew(crew_actor):
    crew=json.loads(crew_actor)
    return [i["name"] for i in crew[:3]]

data["top_3_crew"]= data["crew"].apply(top_3_crew)

In [27]:
data["cast_crew"]= data["top_3_actors"] + data["top_3_crew"]

In [28]:
data["cast_crew"]

0       [Sam Worthington, Zoe Saldana, Sigourney Weave...
1       [Johnny Depp, Orlando Bloom, Keira Knightley, ...
2       [Daniel Craig, Christoph Waltz, Léa Seydoux, T...
3       [Christian Bale, Michael Caine, Gary Oldman, H...
4       [Taylor Kitsch, Lynn Collins, Samantha Morton,...
                              ...                        
4798    [Carlos Gallardo, Jaime de Hoyos, Peter Marqua...
4799    [Edward Burns, Kerry Bishé, Marsha Dietlein, E...
4800    [Eric Mabius, Kristin Booth, Crystal Lowe, Car...
4801    [Daniel Henney, Eliza Coupe, Bill Paxton, Dani...
4802    [Drew Barrymore, Brian Herzlinger, Corey Feldm...
Name: cast_crew, Length: 4803, dtype: object

### 2️⃣ Data Preparation

We created a new column that combines both cast and crew names into a single list.

Then we converted this list into a single string like:
['Sam Worthington', 'Zoe Saldana', 'James Cameron'] -> ['Sam Worthington', 'Zoe Saldana', 'James Cameron']

This gives each movie a text-like representation describing who was involved in it.

In [42]:
data["cast_crew"] = data["cast_crew"].apply(lambda x: " ".join(x) if isinstance(x, list) else str(x))

In [43]:
data["cast_crew"] = data["cast_crew"].apply(lambda x: x.lower().strip() if isinstance(x, str) else "")

In [44]:
data["cast_crew"] = data["cast_crew"].apply(lambda x: ' '.join(x.split()))

In [31]:
print(type(data["cast_crew"].iloc[0]))

<class 'str'>


In [45]:
def clean_text(x):
    if isinstance(x, str):
        # Convert to lowercase
        x = x.lower()
        # Replace all non-alphabetic characters with space
        x = re.sub(r'[^a-z\s]', ' ', x)
        # Remove multiple spaces
        x = re.sub(r'\s+', ' ', x).strip()
        return x
    return ""
    
data["cast_crew"] = data["cast_crew"].apply(clean_text)

In [46]:
data["cast_crew"].head(3)

0    sam worthington zoe saldana sigourney weaver s...
1    johnny depp orlando bloom keira knightley dari...
2    daniel craig christoph waltz l a seydoux thoma...
Name: cast_crew, dtype: object

### 3️⃣ Text Vectorization

Since computers can’t understand text directly, we used a technique called Count Vectorization.

It transforms text into numbers based on how often each word appears.
So, movies with similar names (actors/directors) will have similar number patterns.

This creates a matrix of features — like a fingerprint for every movie.

In [47]:
vectorizer = CountVectorizer()

In [48]:
vectorizer = CountVectorizer(max_features=5000, stop_words='english')
vectors = vectorizer.fit_transform(data["cast_crew"])

In [49]:
print("Vocabulary size:", len(vectorizer.vocabulary_))
print(list(vectorizer.vocabulary_.items())[:20])

Vocabulary size: 5000
[('sam', 4022), ('worthington', 4906), ('zoe', 4991), ('saldana', 4015), ('sigourney', 4235), ('weaver', 4789), ('stephen', 4377), ('rivkin', 3867), ('rick', 3845), ('carter', 742), ('christopher', 853), ('boyes', 552), ('johnny', 2270), ('depp', 1171), ('orlando', 3477), ('bloom', 484), ('keira', 2415), ('knightley', 2505), ('dariusz', 1085), ('wolski', 4888)]


In [50]:
data["cast_crew"].head(10)

0    sam worthington zoe saldana sigourney weaver s...
1    johnny depp orlando bloom keira knightley dari...
2    daniel craig christoph waltz l a seydoux thoma...
3    christian bale michael caine gary oldman hans ...
4    taylor kitsch lynn collins samantha morton and...
5    tobey maguire kirsten dunst james franco franc...
6    zachary levi mandy moore donna murphy john las...
7    robert downey jr chris hemsworth mark ruffalo ...
8    daniel radcliffe rupert grint emma watson brun...
9    ben affleck henry cavill gal gadot hans zimmer...
Name: cast_crew, dtype: object

In [51]:
vectors.shape

(4803, 5000)

In [68]:
print(vectors.toarray()[0][:30])

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [54]:
non_zero_count = np.count_nonzero(vectors.toarray()[0])
print(f"Non-zero values in first movie: {non_zero_count}")
print(f"Words in first movie: {len(data['cast_crew'].iloc[0].split())}")

Non-zero values in first movie: 12
Words in first movie: 13


In [55]:
print(data["cast_crew"].iloc[0])

sam worthington zoe saldana sigourney weaver stephen e rivkin rick carter christopher boyes


### 4️⃣ Calculating Similarity

We then calculated how close two movies are using a mathematical formula called Cosine Similarity.

It measures the angle between two movie vectors — smaller angle = more similarity.
For example:

“Avatar” and “Titanic” may be close because both were directed by James Cameron.

“Avatar” and “The Dark Knight” may be far apart since they have completely different people.

In [57]:
similarity = cosine_similarity(vectors)

In [58]:
def similarity_score(movie1, movie2):
    # Find the index of both movies
    idx1 = data[data['title'].str.lower() == movie1.lower()].index[0]
    idx2 = data[data['title'].str.lower() == movie2.lower()].index[0]
    
    # Return cosine similarity value
    return similarity[idx1][idx2]

In [64]:
def recommend(movie):
    movie_index = data[data['title'].str.lower() == movie.lower()].index[0]
    distances = similarity[movie_index]
    movie_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]
    
    recommended_movies = []
    
    print(f"Movies similar to '{movie}':")
    for i in movie_list:
        #print(data.iloc[i[0]].title)
        recommended_movies.append(data.iloc[i[0]].title)
    return recommended_movies

### 5️⃣ Making Recommendations

We created a function:

In [84]:
movie= "The Matrix"
recommend(movie)

Movies similar to 'The Matrix':


['The Matrix Revolutions',
 'The Matrix Reloaded',
 "Dragon Nest: Warriors' Dawn",
 'Suspect Zero',
 'Red Planet']

In [85]:
similar_movies = recommend(movie)

print("Similarity Scores:")
for sim_movie in similar_movies:
    score = similarity_score(movie, sim_movie)
    print(f"{sim_movie:<10} -->  {score:.3f}")

Movies similar to 'The Matrix':
Similarity Scores:
The Matrix Revolutions -->  0.846
The Matrix Reloaded -->  0.846
Dragon Nest: Warriors' Dawn -->  0.314
Suspect Zero -->  0.240
Red Planet -->  0.231


This looks for all other movies in the dataset, checks which ones have the most similar cast or crew, and lists the top 5.

These movies all share similar directors, writers, or actors.

### 📈 Results (Explained Simply)

The recommendation system successfully finds movies that are connected by people behind them —
it’s like saying:

“If you liked Avatar, you might enjoy Titanic because it was made by the same director, James Cameron.”

This kind of recommendation helps users discover new movies by recognizing patterns in who made them, not just what they’re about.

However:

- It doesn’t yet know the movie genre or story.

- It can’t distinguish between a sci-fi and a romantic movie if both have the same people.

So, it’s accurate for person-based similarity, but not for theme or content-based similarity.

### 💡 Key Takeaways (Non-Technical Summary)
- Vectorization: Turning text into numbers so the computer can compare movies.
- Cosine Similarit: A math way to measure how close two movies are based on people involved.
- Recommendation: Finding movies that share similar cast/crew with the one you liked.